In [2]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [5]:
dataset_path = r"F:\work\python\image frequency\images"
output_path = r"F:\work\python\image frequency\processed_dataset"

train_path = os.path.join(output_path, "train")
test_path = os.path.join(output_path, "test")

categories = ["low", "medium", "high"]

In [6]:
for split in ["train", "test"]:
    for cat in categories:
        os.makedirs(os.path.join(output_path, split, cat), exist_ok=True)

In [7]:
image_paths = []

for class_name in os.listdir(dataset_path):
    class_folder = os.path.join(dataset_path, class_name)
    
    if os.path.isdir(class_folder):
        for img_name in os.listdir(class_folder):
            img_path = os.path.join(class_folder, img_name)
            image_paths.append(img_path)

In [8]:
train_imgs, test_imgs = train_test_split(image_paths, test_size=0.2, random_state=42)

In [12]:
def analyze_image(img):
    # Mean intensity
    mean_intensity = np.mean(img)
    
    # Edge detection
    edges = cv2.Canny(img, 100, 200)
    edge_density = np.sum(edges > 0) / edges.size
    
    return mean_intensity, edge_density

In [9]:
def categorize(mean_intensity, edge_density):
    
    # Intensity thresholds
    if mean_intensity < 85:
        return "low"
    elif mean_intensity < 170:
        return "medium"
    else:
        return "high"

In [10]:
def process_and_save(image_list, split):
    
    for img_path in tqdm(image_list):
        
        # Read image
        img = cv2.imread(img_path)
        
        if img is None:
            continue
        
        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Resize (optional but recommended)
        gray = cv2.resize(gray, (224, 224))
        
        # Analyze
        mean_intensity, edge_density = analyze_image(gray)
        
        # Categorize
        category = categorize(mean_intensity, edge_density)
        
        # Save path
        filename = os.path.basename(img_path)
        save_path = os.path.join(output_path, split, category, filename)
        
        cv2.imwrite(save_path, gray)

In [13]:
process_and_save(train_imgs, "train")
process_and_save(test_imgs, "test")

100%|██████████| 339/339 [00:06<00:00, 54.59it/s]
